# Robust relevance-pursuit GP

この Notebook では、少数の強い外れ値を含むデータに対して `RobustRelevancePursuitSingleTaskGP` と通常の `SingleTaskGP` を比較します。

In [ ]:
import torch
import matplotlib.pyplot as plt
from botorch.fit import fit_gpytorch_mll
from robotorchan.models import SingleTaskGP, RobustRelevancePursuitSingleTaskGP

torch.manual_seed(0)
dtype = torch.double

## 1. 外れ値を含む合成データ

滑らかな正弦波データの一部に大きなずれを加え、外れ値の影響を人工的に作ります。

In [ ]:
train_X = torch.linspace(0, 1, 28, dtype=dtype).unsqueeze(-1)
true_Y = torch.sin(2 * torch.pi * train_X)
train_Y = true_Y + 0.04 * torch.randn_like(true_Y)
outlier_idx = torch.tensor([5, 17, 23])
train_Y[outlier_idx] += torch.tensor([[1.4], [-1.6], [1.2]], dtype=dtype)
train_X.shape, train_Y.shape

## 2. 通常 GP をベースラインとして学習

まず同じデータを通常の `SingleTaskGP` に学習させ、robust GP と比較する基準を作ります。

In [ ]:
baseline = SingleTaskGP(train_X, train_Y)
fit_gpytorch_mll(baseline.make_mll())

## 3. Robust relevance-pursuit GP

次に relevance pursuit を用いる robust GP を学習します。robotorchan wrapper は通常の exact GP と同様に `raw_*` と `make_mll()` を提供します。

In [ ]:
robust = RobustRelevancePursuitSingleTaskGP(
    train_X,
    train_Y,
    convex_parameterization=True,
)
print(robust.raw_train_X.shape, robust.raw_train_Y.shape)
print('supports_mll =', robust.supports_mll)
fit_gpytorch_mll(robust.make_mll())

## 4. Posterior の比較

通常 GP と robust GP の posterior mean を同じグリッドで比較します。

In [ ]:
test_X = torch.linspace(0, 1, 200, dtype=dtype).unsqueeze(-1)
with torch.no_grad():
    p_base = baseline.posterior(test_X)
    p_robust = robust.posterior(test_X)
m_base = p_base.mean.squeeze(-1)
m_robust = p_robust.mean.squeeze(-1)
s_robust = p_robust.variance.sqrt().squeeze(-1)
plt.figure(figsize=(8, 4))
plt.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), label='observations')
plt.scatter(train_X[outlier_idx].squeeze(-1), train_Y[outlier_idx].squeeze(-1), marker='x', s=80, label='outliers')
plt.plot(test_X.squeeze(-1), m_base, label='SingleTaskGP')
plt.plot(test_X.squeeze(-1), m_robust, label='Robust GP')
plt.fill_between(test_X.squeeze(-1), m_robust - 2*s_robust, m_robust + 2*s_robust, alpha=0.2)
plt.legend()
plt.xlabel('x')
plt.ylabel('y')
plt.title('Effect of outliers on GP regression')
plt.show()

## 5. 使い所と注意点

- 少数の観測値が破損している、または異常に強い影響を持つ可能性がある場合に候補になります。
- robust モデルが常に優れるとは限らないため、通常 GP と比較してください。
- 外れ値に見える点が実際には別のレジームなら、mixture、multi-task、conditional model などの方が適切な場合があります。
- robotorchan ではこの wrapper でも標準的な exact-GP の `make_mll()` インターフェースを利用できます。